In [ ]:
%load_ext autoreload
%autoreload 3
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import os, sys
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
import spk_feat_cluster_comp_analysis
importlib.reload(spk_feat_cluster_comp_analysis)
from spk_feat_cluster_comp_analysis import (
    compile_experiment_results,
    plot_within_vs_between_neuron_distances,
    plot_spike_to_avg_distances,
)
import config
from config import SPE1_PICKLE_ROOT

In [ ]:
cluster_pickle_dir = os.path.join(SPE1_PICKLE_ROOT, 'cluster_pickles')
spike_fit_dir      = os.path.join(SPE1_PICKLE_ROOT, 'spike_fit_pickles')

In [ ]:
df_master = compile_experiment_results(cluster_pickle_dir)
df_master['sort_idx'] = df_master['cell_id'].str.extract('(\d+)').astype(int)
df_master = df_master.sort_values(['sort_idx', 'spike_feature']).drop(columns=['sort_idx'])

## Within-cell vs between-cell waveform distances

Do waveform clusters within a single neuron produce differences comparable to differences
between neurons?  If the within-cell cluster distance is in the same range as between-cell
distances, the clustering is picking up biologically meaningful variability — the cell is
as variable internally as it is different from its neighbours.

In [ ]:
plot_within_vs_between_neuron_distances(
    df_master,
    wf_dir=cluster_pickle_dir,
    half_win=75,
    n_bootstrap=2000,
)

In [ ]:
plot_spike_to_avg_distances(
    df_master,
    wf_dir=cluster_pickle_dir,
    spike_fit_dir=spike_fit_dir,
    half_win=75,
)

Between-cell nRMSE reflects the typical waveform difference between any two randomly chosen
neurons in this dataset.  If within-cell cluster nRMSE reaches the lower tail of the
between-cell distribution, the clusters have biological-scale separations.  If within-cell
is entirely below between-cell, the clusters are sub-neuronal variability that doesn't
reach the level of distinguishing cells — which is still informative but means caution is
warranted in interpreting the clusters as discrete functional states.